# 🎮 Minecraft AI Builder - Text-to-Build

Generate Minecraft builds from text prompts like **"medieval castle with towers"**

## ✨ What This Does:

1. Trains AI models (~35-48 hours total)
2. **Generates AI text descriptions for dataset using Gemini** ⚡ REQUIRED
3. Learns text-to-build mapping with CLIP encoder
4. Generates builds from your prompts

## 🚀 How to Use:

1. **🔑 Get Gemini API key** at: https://makersuite.google.com/app/apikey (FREE!)
2. **Enter your API key** in the Configuration cell below
3. **Click Runtime → Run all**
4. **Wait ~35-48 hours** (or pause and resume)
5. **Download generated builds** at the end

## ⚠️ Important:

**Gemini API key is REQUIRED** for this pipeline. The key is used to:
- Generate professional descriptions for BuildPaste dataset
- Analyze build structures (materials, rooms, style)
- Create training data for text-to-build model

Without descriptions, the text-to-build model cannot learn properly.

---
# ⚙️ Configuration

## 🔑 Step 1: Get Your FREE Gemini API Key

1. Go to: **https://makersuite.google.com/app/apikey**
2. Click **"Create API key"**
3. Copy the key (looks like: `AIzaSy...`)
4. Paste it below ⬇️

In [ ]:
# 🔑 PASTE YOUR GEMINI API KEY HERE (REQUIRED)
GEMINI_API_KEY = ""  # ← Put your key between the quotes

# Training configuration
EPOCHS_STAGE_1 = 100  # VQ-VAE epochs (reduce to 50 for faster testing)
EPOCHS_STAGE_2 = 100  # Text-to-build epochs (reduce to 50 for faster testing)
BATCH_SIZE = 4  # Reduce to 2 if out of memory

# ============================================
# VALIDATION - DO NOT MODIFY
# ============================================

import sys

if not GEMINI_API_KEY or GEMINI_API_KEY == "":
    print("❌ ERROR: Gemini API key is REQUIRED!\n")
    print("📝 How to get your FREE API key:\n")
    print("   1. Visit: https://makersuite.google.com/app/apikey")
    print("   2. Click 'Create API key'")
    print("   3. Copy the key")
    print("   4. Paste it in the cell above\n")
    print("⚠️  The API key is FREE and required for:")
    print("   - Generating dataset descriptions")
    print("   - Training text-to-build model")
    print("   - Creating high-quality builds\n")
    raise ValueError("Gemini API key is missing. Please add your key above and run again.")

if len(GEMINI_API_KEY) < 30 or not GEMINI_API_KEY.startswith('AIza'):
    print("❌ ERROR: Invalid API key format!\n")
    print("✓ Valid key should:")
    print("   - Start with 'AIza'")
    print("   - Be 39 characters long")
    print("   - Look like: AIzaSyDxxxxx...\n")
    print(f"Your key: {GEMINI_API_KEY[:10]}... (length: {len(GEMINI_API_KEY)})\n")
    raise ValueError("Invalid API key format. Please check and try again.")

print("✅ Configuration validated!")
print(f"   API Key: {GEMINI_API_KEY[:20]}...{GEMINI_API_KEY[-4:]}")
print(f"   Stage 1 epochs: {EPOCHS_STAGE_1}")
print(f"   Stage 2 epochs: {EPOCHS_STAGE_2}")
print(f"   Batch size: {BATCH_SIZE}")
print("\n🚀 Ready to start training!")

---
# 🚀 Automatic Setup

In [ ]:
import os
os.environ['WANDB_MODE'] = 'disabled'

print("📥 Cloning repository...")
!git clone -b capy/cap-1-d33fde0c https://github.com/GogaGogich123/Ai.git repo
%cd repo/GogaGogich123/Ai
print("✓ Repository cloned and ready")

In [ ]:
print("📦 Installing dependencies...")
!pip install -q -r requirements.txt
!pip install -q -e .
print("✓ Dependencies installed")
print("✓ Package installed")

In [ ]:
import torch
print(f"🔍 System Check:")
print(f"  PyTorch version: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("\n✓ GPU ready for training")
else:
    print("\n⚠️  WARNING: No GPU detected!")
    print("Go to: Runtime → Change runtime type → Hardware accelerator → GPU")
    raise RuntimeError("GPU required for training")

In [ ]:
print("🧪 Testing imports and API...\n")

try:
    from mcbuilder import BuildPasteAPI, ImprovedVQVAE3D
    from mcbuilder import CLIPTextEncoder, TextConditionedLatentDiffusion3D
    from mcbuilder import BLOCKS_ARRAY
    from mcbuilder.gemini_describer import GeminiDescriber
    print(f"✓ All imports successful")
    print(f"✓ {len(BLOCKS_ARRAY)} Minecraft blocks loaded")
except ImportError as e:
    print(f"❌ Import error: {e}")
    raise

# Test BuildPaste API
try:
    api = BuildPasteAPI()
    builds = api.list_builds(limit=3)
    print(f"✓ BuildPaste API working ({len(builds)} builds found)")
    if builds:
        print(f"  Example: {builds[0].name}")
except Exception as e:
    print(f"⚠️  BuildPaste API warning: {e}")
    print("  (Will retry during training)")

# Test Gemini API
print("\n🤖 Testing Gemini API...")
try:
    describer = GeminiDescriber(api_key=GEMINI_API_KEY)
    print("✓ Gemini API client created")
    print("✓ Using model: gemini-1.5-flash (stable, higher quota)")
    print("✓ AI descriptions will be generated during training")
    print("\n💡 Note: API test skipped to preserve quota for training")
except Exception as e:
    print(f"⚠️  Gemini API warning: {e}")
    print("\n📝 If you see quota errors:")
    print("   1. Wait 60 seconds and retry")
    print("   2. Check usage: https://ai.dev/usage?tab=rate-limit")
    print("   3. API key may still work during training (less frequent calls)\n")
    print("⚠️  Continuing anyway - training will handle rate limits...")

print("\n✅ All tests passed! Ready to train.")

---
# 🎯 Stage 1: VQ-VAE + AI Descriptions

**Time:** ~9-14 hours

This stage:
- Downloads builds from BuildPaste
- **Generates AI descriptions with Gemini** 🤖
- Trains VQ-VAE compression model

**Progress will be shown below. You can close the tab and come back later.**

In [ ]:
print("🚀 Starting Stage 1 training with AI descriptions...\n")
print(f"  Epochs: {EPOCHS_STAGE_1}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  AI Descriptions: ENABLED ✓")
print(f"  Gemini API: Connected ✓\n")

!python mcbuilder/train_improved_vqvae.py \
    --cache_dir ./data/cache \
    --checkpoint_dir ./checkpoints_improved \
    --chunk_size 32 \
    --overlap 4 \
    --min_blocks 800 \
    --max_blocks 50000 \
    --batch_size {BATCH_SIZE} \
    --num_workers 2 \
    --embedding_dim 128 \
    --num_embeddings 1024 \
    --num_res_blocks 3 \
    --lr 1e-4 \
    --epochs {EPOCHS_STAGE_1} \
    --save_every 10 \
    --generate_descriptions \
    --gemini_api_key {GEMINI_API_KEY} \
    --description_language en

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("💾 Backing up to Google Drive...")
!mkdir -p /content/drive/MyDrive/minecraft_ai_checkpoints
!cp -r ./checkpoints_improved /content/drive/MyDrive/minecraft_ai_checkpoints/
!cp -r ./data/cache /content/drive/MyDrive/minecraft_ai_checkpoints/

print("\n✓ Stage 1 complete!")
print("✓ Checkpoint saved to Google Drive")
print("✓ AI descriptions generated and cached")

---
# 🎯 Stage 2: Text-Conditioned Diffusion

**Time:** ~15-20 hours

This stage:
- Loads pretrained CLIP text encoder
- Trains diffusion with cross-attention
- Learns to generate from text prompts using AI descriptions

In [ ]:
print(f"🚀 Starting Stage 2 training ({EPOCHS_STAGE_2} epochs)...\n")

!python mcbuilder/train_text_to_build.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --cache_dir ./data/cache \
    --checkpoint_dir ./checkpoints_text_to_build \
    --text_encoder_type clip \
    --context_dim 512 \
    --model_channels 128 \
    --num_res_blocks 2 \
    --attention_resolutions 4 8 \
    --channel_mult 1 2 4 8 \
    --num_heads 8 \
    --dropout 0.1 \
    --timesteps 1000 \
    --chunk_size 32 \
    --batch_size {BATCH_SIZE} \
    --num_workers 2 \
    --epochs {EPOCHS_STAGE_2} \
    --learning_rate 1e-4 \
    --save_every 10

In [ ]:
print("💾 Backing up to Google Drive...")
!cp -r ./checkpoints_text_to_build /content/drive/MyDrive/minecraft_ai_checkpoints/

print("\n✓ Stage 2 complete!")
print("✓ Text-to-build model ready")
print("\n🎉 Training finished! Ready to generate builds!")

---
# ✨ Generate Builds from Text!

Training complete! Now generate builds from prompts.

In [ ]:
print("🏰 Generating: Medieval Castle\n")

!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "medieval stone castle with tall towers and fortified walls" \
    --size 64,48,64 \
    --guidance_scale 8.0 \
    --num_samples 3 \
    --validate \
    --output medieval_castle.litematic

print("\n✓ Castle generated!")

In [ ]:
print("🏡 Generating: Cozy Cottage\n")

!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "cozy cottage with oak planks stone fireplace and wooden furniture" \
    --size 24,20,24 \
    --guidance_scale 7.5 \
    --validate \
    --output cozy_cottage.litematic

print("\n✓ Cottage generated!")

In [ ]:
print("🏢 Generating: Modern House\n")

!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "modern suburban house with white concrete walls and large glass windows" \
    --size 32,24,32 \
    --guidance_scale 7.5 \
    --validate \
    --output modern_house.litematic

print("\n✓ House generated!")

In [ ]:
print("🌳 Generating: Fantasy Treehouse\n")

!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "fantasy treehouse with wooden platforms bridges and leaf decorations" \
    --size 32,40,32 \
    --guidance_scale 8.0 \
    --validate \
    --output treehouse.litematic

print("\n✓ Treehouse generated!")

In [ ]:
print("⛩️ Generating: Japanese Pagoda\n")

!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "traditional japanese pagoda with wooden beams and curved roofs" \
    --size 32,48,32 \
    --guidance_scale 8.5 \
    --validate \
    --output pagoda.litematic

print("\n✓ Pagoda generated!")

---
## 📥 Download Generated Builds

In [ ]:
from google.colab import files
import os

print("📦 Available builds:\n")

litematic_files = [f for f in os.listdir('.') if f.endswith('.litematic')]

if litematic_files:
    for filename in litematic_files:
        size = os.path.getsize(filename) / 1024
        print(f"  📦 {filename} ({size:.1f} KB)")
    
    print("\n📥 Downloading all builds...\n")
    for filename in litematic_files:
        files.download(filename)
    
    print("\n✓ All builds downloaded!")
else:
    print("  No .litematic files found")

print("\n📖 To use in Minecraft:")
print("  1. Install Litematica mod")
print("  2. Place .litematic files in .minecraft/schematics/")
print("  3. Load in-game with M key")

---
## 🎨 Generate Your Own Build

Edit the prompt below and run to generate custom builds!

In [ ]:
# Customize your build here
YOUR_PROMPT = "fantasy wizard tower with magical decorations and bookshelves"
SIZE = "32,48,32"  # X,Y,Z dimensions
GUIDANCE = 8.0     # 1-15, higher = closer to prompt

print(f"🎮 Generating build...")
print(f"  Prompt: {YOUR_PROMPT}")
print(f"  Size: {SIZE}")
print(f"  Guidance: {GUIDANCE}\n")

!python generate_from_text.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --text_to_build_checkpoint ./checkpoints_text_to_build/text_to_build_final.pt \
    --prompt "{YOUR_PROMPT}" \
    --size {SIZE} \
    --guidance_scale {GUIDANCE} \
    --num_samples 3 \
    --validate \
    --output custom_build.litematic

print("\n📥 Downloading...")
files.download('custom_build.litematic')
print("✓ Done!")

---
## 💡 Prompt Examples

**Architecture:**
- `"medieval stone castle with tall towers and fortified walls"`
- `"modern house with glass windows and concrete structure"`
- `"japanese pagoda with wooden beams and curved roofs"`
- `"gothic cathedral with stained glass and stone arches"`

**Fantasy:**
- `"fantasy treehouse with wooden platforms and bridges"`
- `"wizard tower with magical elements and bookshelves"`
- `"elven palace with white marble and nature integration"`
- `"dwarf fortress carved into mountain stone"`

**Functional:**
- `"blacksmith workshop with anvils and furnaces"`
- `"cozy library with bookshelves and reading area"`
- `"medieval tavern with wooden interior and bar"`
- `"enchanting room with magical decorations"`

**Tips:**
- ✅ Be specific about style and materials
- ✅ Mention key features (towers, bridges, windows)
- ✅ Use Minecraft terms (planks, cobblestone, glass)
- ✅ Keep it 5-15 words
- ⚠️ Avoid vague prompts like "house" or "building"

## 🎛️ Parameter Guide

**Size (X,Y,Z):**
- Small: `16,16,16` to `24,24,24` - Fast generation
- Medium: `32,32,32` - Good balance
- Large: `64,48,64` - More detail, slower

**Guidance Scale:**
- Low (3-5): More creative, less accurate
- Medium (7-9): Balanced (recommended)
- High (10-15): Very accurate to prompt

## 📚 Full Documentation

- [TEXT_TO_BUILD.md](https://github.com/GogaGogich123/Ai/blob/capy/cap-1-bc3cdacc/TEXT_TO_BUILD.md) - Complete guide
- [QUICKSTART.md](https://github.com/GogaGogich123/Ai/blob/capy/cap-1-bc3cdacc/QUICKSTART.md) - Training reference
- [README.md](https://github.com/GogaGogich123/Ai/blob/capy/cap-1-bc3cdacc/README.md) - Project overview

---

**Built with ❤️ for the Minecraft community! 🎮✨**

**Powered by Gemini AI for dataset descriptions** 🤖